# Run the Product Evidence Platform
Start Docker Compose from the Azure ML terminal, then use this notebook as a thin client.


In [ ]:
import os, time, requests
AGENT_URL = os.getenv('PRODUCT_AGENT_URL', 'http://127.0.0.1:8788')
health = requests.get(f'{AGENT_URL}/health', timeout=15)
health.raise_for_status()
health.json()


In [ ]:
payload = {
    'product': {
        'row_id': 'ROW-001',
        'main_text': 'Replace with product identity text',
        'country_code': 'CH',
        'retailer_name': None,
        'ean': None,
    },
    'feature_set': 'toy_features',
}
response = requests.post(f'{AGENT_URL}/v1/jobs', json=payload, timeout=30)
response.raise_for_status()
job = response.json()
job


In [ ]:
job_id = job['job_id']
while True:
    status = requests.get(f'{AGENT_URL}/v1/jobs/{job_id}', timeout=15).json()
    print(status['status'], status.get('stage', ''), status.get('message', ''))
    if status['status'] in {'COMPLETED', 'REVIEW_REQUIRED', 'FAILED'}:
        break
    time.sleep(3)


In [ ]:
result_response = requests.get(f'{AGENT_URL}/v1/jobs/{job_id}/result', timeout=30)
result_response.raise_for_status()
result = result_response.json()
result
